<a href="https://colab.research.google.com/github/yukisirosui/AI-Driven-Python-Study-Tracker-with-GitHub-Integration/blob/main/GitHub%E5%90%8C%E6%9C%9F_AI_Python%E3%82%B3%E3%83%BC%E3%83%87%E3%82%A3%E3%83%B3%E3%82%B0%E5%AD%A6%E7%BF%92%E3%83%A1%E3%83%B3%E3%82%BF%E3%83%BC%E3%82%B7%E3%82%B9%E3%83%86%E3%83%A0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔑 実行前の準備：2つのAPIキーの取得と設定

本プログラムを実行するには、**「Gemini APIキー」**と**「GitHubアクセストークン」**が必要です。
閲覧者ご自身のキーが漏洩するのを防ぐため、以下の手順に従って準備をお願いします。

📌 **お知らせ**：GitHubトークンは**未入力（設定なし）のままでも、Gemini AIの基本機能はそのままご利用いただけます。** GitHub連携機能も使用したい場合のみ、手順②の設定を行ってください。

---

## 🛠️ Step 1. 各種キーの取得手順

### 🔹 ① Gemini APIキーの取得（必須 / 無料）
1. [Google AI Studio](https://aistudio.google.com/prompts/new_chat) にアクセスし、Googleアカウントでログインします。
2. 画面左上の **[Get API key]** をクリックします。
3. **[Create API key]** をクリックし、生成されたキー（`AIzaSy...`）をコピーして控えます。

### 🔹 ② GitHub 個人用アクセストークンの取得（任意）
*※GitHub連携を行わない場合は、この手順をスキップして問題ありません。*
1. [GitHub Settings](https://github.com) にログインします。
2. 左側サイドバーの最下部にある **[Developer settings]** をクリックします。
3. **[Personal access tokens]** ＞ **[Fine-grained tokens]**（推奨）を選択します。
4. **[Generate new token]** をクリックし、以下を設定します：
   * **Token name**: 任意の名前（例：`colab-gemini-app`）
   * **Expiration**: 有効期限（推奨：30 days）
   * **Repository access**: 用途に合わせて選択（例：`All repositories`）
   * **Permissions**: コードの実行に必要な権限（例：`Contents: Read and write`）
5. 最下部の **[Generate token]** を押し、表示されたトークン（`github_pat_...`）をコピーします。
   * ⚠️ *画面を閉じると二度と表示されないため、必ずその場でコピーしてください。*

---

## 🔒 Step 2. Google Colabへの安全な設定

取得したキーは、コードに直接書き込まずに、Colabの**「シークレット（鍵マーク）」**機能に保存してプログラムに読み込ませます。

1. 画面左端のメニューバーにある **「鍵のマーク 🔑（シークレット）」** をクリックします。
2. **[新しいシークレットを追加]** をクリックし、必要な項目を登録します。


| 名前（Name） | 値（Value） | ノートブックへのアクセス | 必須 / 任意 |
| :--- | :--- | :---: | :---: |
| **`GEMINI_API_KEY`** | 取得した Gemini の API キー | **ON 🟢** | **必須** |
| **`GITHUB_TOKEN`** | 取得した GitHub のトークン | **ON 🟢** | *任意（空欄でOK）* |

3. 各項目の右側にある **「ノートブックへのアクセス」のスイッチを必ず「ON」** にしてください。


In [73]:
#@title 🤖 Python学習習慣化システム (AIメンター・最新完全版) { display-mode: "form" }
#@markdown 最初にこのセルを実行して、システムを起動させてください。

# ==============================================================================
# 👤 固定設定エリア（配布用URLとトークン）
# ==============================================================================
GITHUB_USER = "yukisirosui"        #@param {type:"string"}
REPO_NAME = "python-daily-study"   #@param {type:"string"}
LOG_FILE = "python_study_log.json" #@param {type:"string"}
MODEL_NAME = "gemini-3.5-flash" # 最新の「gemini-3.5-flash」への書き換えも可能です

# ==============================================================================
# 🎯 学習レベル選択（初学者が選びやすい6段階細分化）
# ==============================================================================
STUDY_LEVEL = "Lv2. 初級（条件分岐・ループ）" #@param ["Lv1. 超初学者（変数・計算）", "Lv2. 初級（条件分岐・ループ）", "Lv3. 初中級（関数・文字列）", "Lv4. 中級（データ構造・例外）", "Lv5. 中上級（アルゴリズム・クラス）", "Lv6. 上級（実践・効率化）"]

# ==============================================================================
# ⚙️ システム初期化
# ==============================================================================
import os
import json
import datetime
import requests
import base64
import time
from google.colab import userdata
from google import genai
from google.genai import types
from datetime import timezone, timedelta

client = None

try:
    # 🔒 APIキーは各自の「シークレット（鍵アイコン）」から安全に自動取得（コードを汚しません）
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
    if GEMINI_API_KEY:
        client = genai.Client(api_key=GEMINI_API_KEY)
        print(f"✅ 最新システム(google.genai)の起動に成功しました！現在の設定レベル: 【{STUDY_LEVEL}】")
        print("👉 下の「🎁 【ステップ1】問題を出す」のセルへ進んでください。")
    else:
        print("【警告】APIキーが取得できません。シークレットの設定を確認してください。")
except Exception as e:
    print("【エラー】ColabのSecretsに 'GEMINI_API_KEY' が正しく設定されているか確認してください。")

current_challenge = None

# ==============================================================================
# 🧠 内部ロジック関数
# ==============================================================================
def _call_gemini_with_retry(prompt, is_json=False):
    """サーバー混雑やデータ破損が起きても自動で3回まで再挑戦する安全ガード機能"""
    for attempt in range(3):
        try:
            config = types.GenerateContentConfig(response_mime_type="application/json") if is_json else None
            response = client.models.generate_content(model=MODEL_NAME, contents=prompt, config=config)
            if not response.text:
                raise ValueError("AIからの返答が空っぽです")
            if is_json:
                return json.loads(response.text)
            return response.text
        except Exception as e:
            if attempt < 2:
                print(f"⏳ AIサーバー混雑またはデータ乱れを検知。3秒後に自動再挑戦します... (試行 {attempt+1}/3)")
                time.sleep(3)
            else:
                print("\n❌ AIサーバーが現在非常に混雑しています。1分ほど置いてから再度お試しください。")
                raise e

def _internal_generate():
    global current_challenge
    if not client:
        print("❌ Gemini APIキーが設定されていないため、問題を生成できません。")
        return
    history_summary = "なし"
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, "r", encoding="utf-8") as f:
            try:
                logs = json.load(f)
                past_topics = []
                for day in logs:
                    for ch in day.get("challenges", []):
                        past_topics.append(ch.get("topic", ""))
                history_summary = ", ".join(past_topics[-5:])
            except: pass

    prompt = f"""
    AIプログラミングメンターとして、学習者の指定レベルに完璧に合致するコーディング課題を1問作成してください。
    【指定レベル】: {STUDY_LEVEL}
    【過去のテーマ】: {history_summary}
    【出力フォーマット】必ずJSON形式でのみ出力。
    {{
        "title": "課題のタイトル",
        "topic": "学習テーマ",
        "description": "問題の具体的な説明",
        "example_input": "入力例",
        "example_output": "期待される出力例"
    }}
    """
    try:
        current_challenge = _call_gemini_with_retry(prompt, is_json=True)
        print(f"🎯 【タイトル】: {current_challenge['title']}\n")
        print(f"📚 【テーマ】: {current_challenge['topic']}\n")
        print(f"📝 【問題内容】:\n{current_challenge['description']}\n")
        print(f"📥 【入力例】: {current_challenge['example_input']}")
        print(f"📤 【出力例】: {current_challenge['example_output']}\n")
        print("="*60)
        print("💡 【次のステップ】\n1. わからない時は「🤔 【ステップ2】ヒントをもらう」を実行\n2. 書けたら「🩺 【ステップ3】提出・レビュー」へ！")
    except: pass

def _internal_hint():
    if not client or not current_challenge: return
    print("🤔 メンターがヒントを考えています...")
    prompt = f"解答コードそのものは絶対に明かさず、実装の手がかりとなるヒントを3段階で優しく教えてください。\n課題: {current_challenge['title']}\n内容: {current_challenge['description']}"
    try:
        hint_text = _call_gemini_with_retry(prompt, is_json=False)
        print(f"\n💡 【メンターからのヒント】\n{hint_text}")
    except: pass

def _internal_review(user_code):
    if not client or not current_challenge: return
    if not user_code or user_code.strip() == "# ここにあなたの回答コードを書いてください":
        print("⚠️ コードが入力されていません。")
        return
    print("\n🩺 AIメディックがコードを診断中...")
    prompt = f"""
    出題した課題に対して、学習者が書いたコードをレビューし、採点とアドバイスをしてください。
    【課題】{current_challenge['title']} / {current_challenge['description']}
    【コード】\n```python\n{user_code}\n```
    【出力フォーマット】必ずJSON形式でのみ出力。
    {{
        "status": "「クリア」または「要修正」",
        "score": 100点満点中の点数(数値),
        "review": "コードのアドバイス",
        "refactored_code": "60点以上の場合は見本コード。60点未満の場合は空欄"
    }}
    """
    try:
        result = _call_gemini_with_retry(prompt, is_json=True)
        print(f"\n📊 採点結果: {result['score']}点 【{result['status']}】")
        print(f"💬 メンターのアドバイス:\n{result['review']}")

        if result['score'] < 60:
            print("\n⚠️ 【判定: 不合格】再挑戦しましょう！")
            return
        if result['refactored_code']:
            print(f"\n💡 模範コード（リファクタリング案）:\n{result['refactored_code']}")
        _internal_save_and_push(user_code, result['score'])
    except: pass

def _internal_save_and_push(user_code, score):
    today_str = str(datetime.date.today())
    logs = []
    if os.path.exists(LOG_FILE):
        with open(LOG_FILE, "r", encoding="utf-8") as f:
            try: logs = json.load(f)
            except: pass

     # 📝 サーバーの位置に関係なく、強制的に日本時間にする設定
    from datetime import timezone, timedelta
    jst = timezone(timedelta(hours=+9))
    now_jst = datetime.datetime.now(jst)

    challenge_data = {
        "time": now_jst.strftime("%H:%M:%S"), # これで確実に日本の24時間表記になります
        "title": current_challenge['title'],
        "topic": current_challenge['topic'],
        "problem_description": current_challenge['description'],
        "my_code": user_code,
        "score": score
    }


    existing_day_index = -1
    for i, log in enumerate(logs):
        if log.get("date") == today_str:
            existing_day_index = i
            break

    if existing_day_index != -1:
        if not isinstance(logs[existing_day_index], dict) or "challenges" not in logs[existing_day_index]:
            logs[existing_day_index] = {"date": today_str, "total_solved_today": 0, "challenges": []}
        logs[existing_day_index]["challenges"].append(challenge_data)
        logs[existing_day_index]["total_solved_today"] = len(logs[existing_day_index]["challenges"])
        print(f"\n🔄 本日（{today_str}）の{logs[existing_day_index]['total_solved_today']}問目の合格コードを履歴に追加しました！")
        is_new_day = False
    else:
        new_day_data = { "date": today_str, "total_solved_today": 1, "challenges": [challenge_data] }
        logs.append(new_day_data)
        print(f"\n🔥 今日の最初の学習スタンプを記録しました！ (累計クリア日数: {len(logs)}日)")
        is_new_day = True

    content_str = json.dumps(logs, ensure_ascii=False, indent=4)
    with open(LOG_FILE, "w", encoding="utf-8") as f:
        f.write(content_str)

    try:
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        token = None

    if not token:
        print("ℹ️ GITHUB_TOKENが未設定のため、GitHub同期をスキップしました。")
        return

    try:
        url = f"https://api.github.com/repos/{GITHUB_USER}/{REPO_NAME}/contents/{LOG_FILE}"
        headers = {"Authorization": f"token {token}", "Accept": "application/vnd.github.v3+json"}

        # 404（リリポジトリにまだファイルがない初回状態）を安全に検知する処理
        res = requests.get(url, headers=headers)
        if res.status_code == 200:
            sha = res.json().get("sha")
        elif res.status_code == 404:
            sha = None
        else:
            print(f"⚠️ GitHubファイル状態の取得中に異常を検知しました (Status: {res.status_code})")
            sha = None

        commit_msg = f"🌱 Add New Day: {today_str}" if is_new_day else f"🚀 Add Another Solved Challenge: {today_str}"
        data = {
            "message": f"🤖 {commit_msg} ({current_challenge['title']})",
            "content": base64.b64encode(content_str.encode("utf-8")).decode("utf-8")
        }
        if sha: data["sha"] = sha
        put_res = requests.put(url, headers=headers, json=data)

        # 判定用リストによる綺麗な文法に修正済み
        success_codes = [200, 201]
        if put_res.status_code in success_codes:
            print(f"✨ GitHubとの同期に成功！今日の合格コードがすべて安全に保存されました🌱")
        else:
            print(f"⚠️ GitHub同期エラー: {put_res.json().get('message')} (Status: {put_res.status_code})")
    except Exception as e:
        print(f"❌ GitHub連携中にエラーが発生しました: {e}")


In [72]:
# 🎁 【ステップ1】問題を出す
_internal_generate()


In [64]:
# 🤔 【ステップ2】ヒントをもらう
# どうしても書き方がわからない時は、このセルを実行すると3段階のヒントが出ます
_internal_hint()


In [71]:
# 🩺 【ステップ3】提出・レビュー
# 以下のトリプルクォーテーション（ """ ）の間に、あなたの回答コードを貼り付けて実行してください。

my_answer = """

# ここにあなたの回答コードを書いてください
# （例）
# price = int(input())
# print(price * 1.1)


"""

# AIメンターへ提出
_internal_review(my_answer)


In [ ]:
# ここにあなたの回答コードを書いてください
# （例）
# price = int(input())
# print(price * 1.1)

In [19]:
#@title 🔍 オプション: AI辞書フォーム { display-mode: "form" }
調べたい単語 = "`try-except`" #@param {type:"string"}

prompt = f"""
あなたは親切なPythonプログラミング講師です。「{調べたい単語}」について解説してください。
1. 【一言でいうと】: 簡単な説明
2. 【詳しい仕組み】: Pythonにおける役割
3. 【具体的なコード例】: 短いコードと実行結果
"""
print(f"🔍 『{調べたい単語}』についてAIメンターが調べています...")
response = model.generate_content(prompt)
print("\n" + "="*50 + f"\n📖 【AI辞書】単語解説: {調べたい単語}\n" + "="*50)
print(response.text)


#過去のコード（検証用）

In [60]:
# @title
# #@title 🤖 Python学習習慣化システム (AIメンター・GitHub維持版) { display-mode: "form" }
# #@markdown 最初にこのセルを実行して、システムを起動させてください。
# #@markdown （※実行前に、左メニューの「鍵アイコン」から `GEMINI_API_KEY` を登録してください）

# # ==============================================================================
# # 👤 固定設定エリア（配布用URLとトークン）
# # ==============================================================================
# GITHUB_USER = "yukisirosui"        #@param {type:"string"}
# REPO_NAME = "python-daily-study"   #@param {type:"string"}
# LOG_FILE = "python_study_log.json" #@param {type:"string"}
# MODEL_NAME = "models/gemini-3.5-flash" #@param {type:"string"}

# # ==============================================================================
# # 🎯 学習レベル選択
# # ==============================================================================
# STUDY_LEVEL = "Lv2. 初級（条件分岐・ループ）" #@param ["Lv1. 超初学者（変数・計算）", "Lv2. 初級（条件分岐・ループ）", "Lv3. 初中級（関数・文字列）", "Lv4. 中級（データ構造・例外）", "Lv5. 中上級（アルゴリズム・クラス）", "Lv6. 上級（実践・効率化）"]

# # ==============================================================================
# # ⚙️ システム初期化
# # ==============================================================================
# import os
# import json
# import datetime
# import requests
# import base64
# from google.colab import userdata
# from google import genai
# from google.genai import types

# client = None

# try:
#     # 🔒 APIキーはColabの「シークレット（鍵アイコン）」から安全に自動取得します
#     GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

#     if GEMINI_API_KEY:
#         client = genai.Client(api_key=GEMINI_API_KEY)
#         print(f"✅ 最新システム(google.genai)の起動に成功しました！現在の設定レベル: 【{STUDY_LEVEL}】")
#         print("👉 下の「🎁 【ステップ1】問題を出す」のセルへ進んでください。")
#     else:
#         print("【警告】APIキーが取得できません。シークレットの設定を確認してください。")
# except Exception as e:
#     print("【エラー】ColabのSecrets(左メニューの鍵アイコン)に 'GEMINI_API_KEY' が正しく設定されているか確認してください。")

# # 全体で共有する課題用変数
# current_challenge = None

# # ==============================================================================
# # 🧠 内部ロジック関数
# # ==============================================================================
# def _internal_generate():
#     global current_challenge
#     if not client:
#         print("❌ Gemini APIキーが設定されていないため、問題を生成できません。")
#         return

#     history_summary = "なし"
#     if os.path.exists(LOG_FILE):
#         with open(LOG_FILE, "r", encoding="utf-8") as f:
#             try:
#                 logs = json.load(f)
#                 past_topics = []
#                 for day in logs:
#                     for ch in day.get("challenges", []):
#                         past_topics.append(ch.get("topic", ""))
#                 history_summary = ", ".join(past_topics[-5:])
#             except: pass

#     prompt = f"""
#     AIプログラミングメンターとして、学習者の指定レベルに完璧に合致するコーディング課題を1問作成してください。
#     【指定レベル】: {STUDY_LEVEL}
#     【過去のテーマ】: {history_summary}
#     【出力フォーマット】必ずJSON形式でのみ出力。
#     {{
#         "title": "課題のタイトル",
#         "topic": "学習テーマ",
#         "description": "問題の具体的な説明",
#         "example_input": "入力例",
#         "example_output": "期待される出力例"
#     }}
#     """
#     try:
#         response = client.models.generate_content(
#             model=MODEL_NAME, contents=prompt,
#             config=types.GenerateContentConfig(response_mime_type="application/json"),
#         )
#         current_challenge = json.loads(response.text)
#         print(f"🎯 【タイトル】: {current_challenge['title']}\n")
#         print(f"📚 【テーマ】: {current_challenge['topic']}\n")
#         print(f"📝 【問題内容】:\n{current_challenge['description']}\n")
#         print(f"📥 【入力例】: {current_challenge['example_input']}")
#         print(f"📤 【出力例】: {current_challenge['example_output']}\n")
#         print("="*60)
#         print("💡 【次のステップ】\n1. わからない時は「🤔 【ステップ2】ヒントをもらう」を実行\n2. 書けたら「🩺 【ステップ3】提出・レビュー」へ！")
#     except Exception as e:
#         print(f"❌ 問題生成中にエラーが発生しました: {e}")

# def _internal_hint():
#     if not client: return
#     if not current_challenge:
#         print("⚠️ まだ問題が生成されていません。先に【ステップ1】を実行してください。")
#         return
#     print("🤔 メンターがヒントを考えています...")
#     prompt = f"解答コードそのものは絶対に明かさず、実装の手がかりとなるヒントを3段階で優しく教えてください。\n課題: {current_challenge['title']}\n内容: {current_challenge['description']}"
#     response = client.models.generate_content(model=MODEL_NAME, contents=prompt)
#     print(f"\n💡 【メンターからのヒント】\n{response.text}")

# def _internal_review(user_code):
#     if not client: return
#     if not current_challenge:
#         print("⚠️ まだ問題が生成されていません。先に【ステップ1】を実行してください。")
#         return
#     if not user_code or user_code.strip() == "# ここにあなたの回答コードを書いてください":
#         print("⚠️ コードが入力されていません。")
#         return

#     print("\n🩺 AIメディックがコードを診断中...")
#     prompt = f"""
#     出題した課題に対して、学習者が書いたコードをレビューし、採点とアドバイスをしてください。
#     【課題】{current_challenge['title']} / {current_challenge['description']}
#     【コード】\n```python\n{user_code}\n```
#     【出力フォーマット】必ずJSON形式でのみ出力。
#     {{
#         "status": "「クリア」または「要修正」",
#         "score": 100点満点中の点数(数値),
#         "review": "コードのアドバイス",
#         "refactored_code": "60点以上の場合は見本コード。60点未満の場合は空欄"
#     }}
#     """
#     try:
#         response = client.models.generate_content(
#             model=MODEL_NAME, contents=prompt,
#             config=types.GenerateContentConfig(response_mime_type="application/json"),
#         )
#         result = json.loads(response.text)
#         print(f"\n📊 採点結果: {result['score']}点 【{result['status']}】")
#         print(f"💬 メンターのアドバイス:\n{result['review']}")

#         if result['score'] < 60:
#             print("\n⚠️ 【判定: 不合格】再挑戦しましょう！")
#             return
#         if result['refactored_code']:
#             print(f"\n💡 模範コード（リファクタリング案）:\n{result['refactored_code']}")

#         # 合格した場合のみ、以前のGitHubプッシュ処理を呼び出す
#         _internal_save_and_push(user_code, result['score'])
#     except Exception as e:
#         print(f"❌ レビュー中にエラーが発生しました: {e}")

# def _internal_save_and_push(user_code, score):
#     today_str = str(datetime.date.today())
#     logs = []
#     if os.path.exists(LOG_FILE):
#         with open(LOG_FILE, "r", encoding="utf-8") as f:
#             try: logs = json.load(f)
#             except: pass

#     challenge_data = {
#         "time": datetime.datetime.now().strftime("%H:%M:%S"),
#         "title": current_challenge['title'], "topic": current_challenge['topic'],
#         "problem_description": current_challenge['description'], "my_code": user_code, "score": score
#     }

#     existing_day_index = -1
#     for i, log in enumerate(logs):
#         if log.get("date") == today_str:
#             existing_day_index = i
#             break

#     if existing_day_index != -1:
#         if not isinstance(logs[existing_day_index], dict) or "challenges" not in logs[existing_day_index]:
#             logs[existing_day_index] = {"date": today_str, "total_solved_today": 0, "challenges": []}
#         logs[existing_day_index]["challenges"].append(challenge_data)
#         logs[existing_day_index]["total_solved_today"] = len(logs[existing_day_index]["challenges"])
#         print(f"\n🔄 本日（{today_str}）の{logs[existing_day_index]['total_solved_today']}問目の合格コードを履歴に追加しました！")
#         is_new_day = False
#     else:
#         new_day_data = { "date": today_str, "total_solved_today": 1, "challenges": [challenge_data] }
#         logs.append(new_day_data)
#         print(f"\n🔥 今日の最初の学習スタンプを記録しました！ (累計クリア日数: {len(logs)}日)")
#         is_new_day = True

#     content_str = json.dumps(logs, ensure_ascii=False, indent=4)
#     with open(LOG_FILE, "w", encoding="utf-8") as f:
#         f.write(content_str)

#     # 🔑 以前の仕組み通り、ColabのシークレットからGITHUB_TOKENを取得します
#     try:
#         token = userdata.get('GITHUB_TOKEN')
#     except Exception:
#         token = None

#     if not token:
#         print("ℹ️ GITHUB_TOKENがColabのシークレットに設定されていないため、GitHub同期をスキップしました。")
#         return

#     try:
#         url = f"https://api.github.com/{GITHUB_USER}/{REPO_NAME}/contents/{LOG_FILE}"
#         headers = {"Authorization": f"token {token}", "Accept": "application/vnd.github.v3+json"}

#         res = requests.get(url, headers=headers)
#         sha = res.json().get("sha") if res.status_code == 200 else None

#         commit_msg = f"🌱 Add New Day: {today_str}" if is_new_day else f"🚀 Add Another Solved Challenge: {today_str}"
#         data = {
#             "message": f"🤖 {commit_msg} ({current_challenge['title']})",
#             "content": base64.b64encode(content_str.encode("utf-8")).decode("utf-8")
#         }
#         if sha: data["sha"] = sha
#         put_res = requests.put(url, headers=headers, json=data)

#         if put_res.status_code in [200, 201]:
#             print(f"✨ GitHubとの同期に成功！今日の合格コードがすべて安全に保存されました🌱")
#         else:
#             print(f"⚠️ GitHub同期エラー: {put_res.json().get('message')}")
#     except Exception as e:
#         print(f"❌ GitHub連携中にエラーが発生しました: {e}")


In [25]:
# @title
## 付録：過去のコード（検証用）

In [21]:
# @title
# #@title 🤖 Python学習習慣化システム (AIメンター・フォーム確定版) { display-mode: "form" }
# #@markdown 最初にこのセルを実行して、システムを起動させてください。

# import os
# import json
# import datetime
# import requests
# import base64
# from google.colab import userdata
# import google.generativeai as genai

# try:
#     GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
#     genai.configure(api_key=GOOGLE_API_KEY)
#     model = genai.GenerativeModel('models/gemini-2.5-flash')
#     print("✅ システムの起動に成功しました！下の各コントロールフォームへ進んでください。")
# except Exception as e:
#     print("【エラー】ColabのSecretsに 'GEMINI_API_KEY' が設定されているか確認してください。")

# LOG_FILE = "python_study_log.json"

# def _internal_generate(level):
#     history_summary = "なし"
#     if os.path.exists(LOG_FILE):
#         with open(LOG_FILE, "r", encoding="utf-8") as f:
#             try:
#                 logs = json.load(f)
#                 past_topics = []
#                 for day in logs:
#                     for ch in day.get("challenges", []):
#                         past_topics.append(ch.get("topic", ""))
#                 history_summary = ", ".join(past_topics[-5:])
#             except: pass

#     prompt = f"""
#     AIプログラミングメンターとして、学習者の条件（レベル: {level}、最近解いたテーマ: {history_summary}）に合う課題を1問作成してください。
#     【出力フォーマット】必ず以下のJSON形式でのみ出力してください。
#     {{
#         "title": "課題のタイトル",
#         "topic": "学習テーマ",
#         "description": "問題の具体的な説明",
#         "example_input": "入力例",
#         "example_output": "期待される出力例"
#     }}
#     """
#     response = model.generate_content(prompt, generation_config={"response_mime_type": "application/json"})
#     return json.loads(response.text)

# def _internal_hint(challenge):
#     print("🤔 メンターがヒントを考えています...")
#     prompt = f"""
#     学習者が以下の課題で行き詰まっています。解答コードそのものは絶対に明かさず、
#     実装の手がかりとなるヒントを3段階で優しく教えてください。
#     課題: {challenge['title']}\n内容: {challenge['description']}
#     """
#     response = model.generate_content(prompt)
#     print(f"\n💡 【メンターからのヒント】\n{response.text}")

# def _internal_review(challenge, user_code):
#     print("\n🩺 AIメディックがコードを診断中...")
#     prompt = f"""
#     出題した課題に対して、学習者が書いたコードをレビューし、採点とアドバイスをしてください。
#     【課題】{challenge['title']} / {challenge['description']}
#     【コード】\n```python\n{user_code}\n```
#     【出力フォーマット】必ず以下のJSON形式でのみ出力してください。
#     {{
#         "status": "「クリア」または「要修正」",
#         "score": 100点満点中の点数(数値),
#         "review": "コードのアドバイス",
#         "refactored_code": "60点以上の場合は見本コード。60点未満の場合は空欄"
#     }}
#     """
#     response = model.generate_content(prompt, generation_config={"response_mime_type": "application/json"})
#     result = json.loads(response.text)

#     print(f"\n📊 採点結果: {result['score']}点 【{result['status']}】")
#     print(f"💬 メンターのアドバイス:\n{result['review']}")

#     if result['score'] < 60:
#         print("\n⚠️ 【判定: 不合格】点数が60点に満たなかったため、今回のコードは記録されません。")
#         return

#     if result['refactored_code']:
#         print(f"\n💡 模範コード（リファクタリング案）:\n{result['refactored_code']}")

#     _internal_save_and_push(challenge, user_code, result['score'])

# def _internal_save_and_push(challenge, user_code, score):
#     today_str = str(datetime.date.today())
#     logs = []
#     if os.path.exists(LOG_FILE):
#         with open(LOG_FILE, "r", encoding="utf-8") as f:
#             try: logs = json.load(f)
#             except: pass

#     challenge_data = {
#         "time": datetime.datetime.now().strftime("%H:%M:%S"),
#         "title": challenge['title'],
#         "topic": challenge['topic'],
#         "problem_description": challenge['description'],
#         "my_code": user_code,
#         "score": score
#     }

#     existing_day_index = -1
#     for i, log in enumerate(logs):
#         if log.get("date") == today_str:
#             existing_day_index = i
#             break

#     if existing_day_index != -1:
#         if not isinstance(logs[existing_day_index], dict) or "challenges" not in logs[existing_day_index]:
#             logs[existing_day_index] = {"date": today_str, "total_solved_today": 0, "challenges": []}
#         logs[existing_day_index]["challenges"].append(challenge_data)
#         logs[existing_day_index]["total_solved_today"] = len(logs[existing_day_index]["challenges"])
#         print(f"\n🔄 本日（{today_str}）の{logs[existing_day_index]['total_solved_today']}問目の合格コードを履歴に追加しました！")
#         is_new_day = False
#     else:
#         new_day_data = {
#             "date": today_str,
#             "total_solved_today": 1,
#             "challenges": [challenge_data]
#         }
#         logs.append(new_day_data)
#         print(f"\n🔥 今日の最初の学習スタンプを記録しました！ (累計クリア日数: {len(logs)}日)")
#         is_new_day = True

#     content_str = json.dumps(logs, ensure_ascii=False, indent=4)
#     with open(LOG_FILE, "w", encoding="utf-8") as f:
#         f.write(content_str)

#     try:
#         GITHUB_USER = "yukisirosui"
#         REPO_NAME = "python-daily-study"  # ⭕ あなたの実際のGitHubリポジトリ名に修正
#         token = userdata.get('GITHUB_TOKEN')

#         # ⭕ APIのURLには「repos」が必要です
#         url = f"https://api.github.com/repos/{GITHUB_USER}/{REPO_NAME}/contents/{LOG_FILE}"

#         headers = {"Authorization": f"token {token}", "Accept": "application/vnd.github.v3+json"}

#         res = requests.get(url, headers=headers)
#         sha = res.json().get("sha") if res.status_code == 200 else None

#         commit_msg = f"🌱 Add New Day: {today_str}" if is_new_day else f"🚀 Add Another Solved Challenge: {today_str}"
#         data = {
#             "message": f"🤖 {commit_msg} ({challenge['title']})",
#             "content": base64.b64encode(content_str.encode("utf-8")).decode("utf-8")
#         }
#         if sha: data["sha"] = sha
#         put_res = requests.put(url, headers=headers, json=data)

#         # 修正：成功ステータス（200番台）を正しく判定
#         if put_res.status_code in [200, 201]:
#             print(f"✨ GitHubとの同期に成功！今日の合格コードがすべて安全に保存されました🌱")
#         else:
#             print(f"⚠️ GitHub同期エラー: {put_res.json().get('message')}")
#     except Exception as e:
#         print(f"❌ GitHub連携中にエラーが発生しました: {e}")

In [22]:
# @title
# #@title 🤖 Python学習習慣化システム (AIメンター・フォーム確定版) { display-mode: "form" }
# #@markdown 最初に以下の「ユーザー個人設定」を確認・変更してから、このセルを実行してください。

# # ==============================================================================
# # 👤 ユーザー個人設定エリア
# # ==============================================================================
# GITHUB_USER = "yukisirosui"        #@param {type:"string"}
# REPO_NAME = "python-daily-study"   #@param {type:"string"}
# LOG_FILE = "python_study_log.json" #@param {type:"string"}
# MODEL_NAME = "models/gemini-2.5-flash" #@param {type:"string"}

# # ==============================================================================
# # ⚙️ システム初期化
# # ==============================================================================
# import os
# import json
# import datetime
# import requests
# import base64
# from google.colab import userdata
# import google.generativeai as genai

# try:
#     # Google Colabの「Secrets（鍵アイコン）」からAPIキーを取得します
#     GOOGLE_API_KEY = userdata.get('GEMINI_API_KEY')
#     genai.configure(api_key=GOOGLE_API_KEY)
#     model = genai.GenerativeModel(MODEL_NAME)
#     print("✅ システムの起動に成功しました！下の各コントロールフォームへ進んでください。")
# except Exception as e:
#     print("【エラー】ColabのSecrets(左メニューの鍵アイコン)に 'GEMINI_API_KEY' が正しく設定されているか確認してください。")

# # ==============================================================================
# # 🧠 内部ロジック関数
# # ==============================================================================
# def _internal_generate(level):
#     history_summary = "なし"
#     if os.path.exists(LOG_FILE):
#         with open(LOG_FILE, "r", encoding="utf-8") as f:
#             try:
#                 logs = json.load(f)
#                 past_topics = []
#                 for day in logs:
#                     for ch in day.get("challenges", []):
#                         past_topics.append(ch.get("topic", ""))
#                 history_summary = ", ".join(past_topics[-5:])
#             except: pass

#     prompt = f"""
#     AIプログラミングメンターとして、学習者の条件（レベル: {level}、最近解いたテーマ: {history_summary}）に合う課題を1問作成してください。
#     【出力フォーマット】必ず以下のJSON形式でのみ出力してください。
#     {{
#         "title": "課題のタイトル",
#         "topic": "学習テーマ",
#         "description": "問題の具体的な説明",
#         "example_input": "入力例",
#         "example_output": "期待される出力例"
#     }}
#     """
#     response = model.generate_content(prompt, generation_config={"response_mime_type": "application/json"})
#     return json.loads(response.text)

# def _internal_hint(challenge):
#     print("🤔 メンターがヒントを考えています...")
#     prompt = f"""
#     学習者が以下の課題で行き詰まっています。解答コードそのものは絶対に明かさず、
#     実装の手がかりとなるヒントを3段階で優しく教えてください。
#     課題: {challenge['title']}\n内容: {challenge['description']}
#     """
#     response = model.generate_content(prompt)
#     print(f"\n💡 【メンターからのヒント】\n{response.text}")

# def _internal_review(challenge, user_code):
#     print("\n🩺 AIメディックがコードを診断中...")
#     prompt = f"""
#     出題した課題に対して、学習者が書いたコードをレビューし、採点とアドバイスをしてください。
#     【課題】{challenge['title']} / {challenge['description']}
#     【コード】\n```python\n{user_code}\n```
#     【出力フォーマット】必ず以下のJSON形式でのみ出力してください。
#     {{
#         "status": "「クリア」または「要修正」",
#         "score": 100点満点中の点数(数値),
#         "review": "コードのアドバイス",
#         "refactored_code": "60点以上の場合は見本コード。60点未満の場合は空欄"
#     }}
#     """
#     response = model.generate_content(prompt, generation_config={"response_mime_type": "application/json"})
#     result = json.loads(response.text)

#     print(f"\n📊 採点結果: {result['score']}点 【{result['status']}】")
#     print(f"💬 メンターのアドバイス:\n{result['review']}")

#     if result['score'] < 60:
#         print("\n⚠️ 【判定: 不合格】点数が60点に満たなかったため、今回のコードは記録されません。")
#         return

#     if result['refactored_code']:
#         print(f"\n💡 模範コード（リファクタリング案）:\n{result['refactored_code']}")

#     _internal_save_and_push(challenge, user_code, result['score'])

# def _internal_save_and_push(challenge, user_code, score):
#     today_str = str(datetime.date.today())
#     logs = []
#     if os.path.exists(LOG_FILE):
#         with open(LOG_FILE, "r", encoding="utf-8") as f:
#             try: logs = json.load(f)
#             except: pass

#     challenge_data = {
#         "time": datetime.datetime.now().strftime("%H:%M:%S"),
#         "title": challenge['title'],
#         "topic": challenge['topic'],
#         "problem_description": challenge['description'],
#         "my_code": user_code,
#         "score": score
#     }

#     existing_day_index = -1
#     for i, log in enumerate(logs):
#         if log.get("date") == today_str:
#             existing_day_index = i
#             break

#     if existing_day_index != -1:
#         if not isinstance(logs[existing_day_index], dict) or "challenges" not in logs[existing_day_index]:
#             logs[existing_day_index] = {"date": today_str, "total_solved_today": 0, "challenges": []}
#         logs[existing_day_index]["challenges"].append(challenge_data)
#         logs[existing_day_index]["total_solved_today"] = len(logs[existing_day_index]["challenges"])
#         print(f"\n🔄 本日（{today_str}）の{logs[existing_day_index]['total_solved_today']}問目の合格コードを履歴に追加しました！")
#         is_new_day = False
#     else:
#         new_day_data = {
#             "date": today_str,
#             "total_solved_today": 1,
#             "challenges": [challenge_data]
#         }
#         logs.append(new_day_data)
#         print(f"\n🔥 今日の最初の学習スタンプを記録しました！ (累計クリア日数: {len(logs)}日)")
#         is_new_day = True

#     content_str = json.dumps(logs, ensure_ascii=False, indent=4)
#     with open(LOG_FILE, "w", encoding="utf-8") as f:
#         f.write(content_str)

#     try:
#         token = userdata.get('GITHUB_TOKEN')
#         url = f"https://api.github.com/repos/{GITHUB_USER}/{REPO_NAME}/contents/{LOG_FILE}"
#         headers = {"Authorization": f"token {token}", "Accept": "application/vnd.github.v3+json"}

#         res = requests.get(url, headers=headers)
#         sha = res.json().get("sha") if res.status_code == 200 else None

#         commit_msg = f"🌱 Add New Day: {today_str}" if is_new_day else f"🚀 Add Another Solved Challenge: {today_str}"
#         data = {
#             "message": f"🤖 {commit_msg} ({challenge['title']})",
#             "content": base64.b64encode(content_str.encode("utf-8")).decode("utf-8")
#         }
#         if sha: data["sha"] = sha
#         put_res = requests.put(url, headers=headers, json=data)

#         if put_res.status_code in [200, 201]:
#             print(f"✨ GitHubとの同期に成功！今日の合格コードがすべて安全に保存されました🌱")
#         else:
#             print(f"⚠️ GitHub同期エラー: {put_res.json().get('message')}")
#     except Exception as e:
#         print(f"❌ GitHub連携中にエラーが発生しました: {e}")


In [20]:
# @title
# #@title 🤖 Python学習習慣化システム (AIメンター・最新SDK対応版) { display-mode: "form" }
# #@markdown 最初に以下の「ユーザー個人設定」を確認・変更してから、このセルを実行してください。

# # ==============================================================================
# # 👤 ユーザー個人設定エリア
# # ==============================================================================
# #@markdown #### 🔑 API・トークン設定（未入力の場合はColabの「シークレット」から自動取得を試みます）
# GEMINI_API_KEY = "" #@param {type:"username"}
# GITHUB_TOKEN = "" #@param {type:"username"}

# #@markdown #### 📂 GitHub・ファイル設定
# GITHUB_USER = "yukisirosui"        #@param {type:"string"}
# REPO_NAME = "python-daily-study"   #@param {type:"string"}
# LOG_FILE = "python_study_log.json" #@param {type:"string"}
# MODEL_NAME = "gemini-2.5-flash"    #@param {type:"string"}

# # ==============================================================================
# # ⚙️ システム初期化
# # ==============================================================================
# import os
# import json
# import datetime
# import requests
# import base64
# from google.colab import userdata

# # 📝 新しいパッケージ「google.genai」をインポート
# from google import genai
# from google.genai import types

# # --- APIキーの取得ロジック（画面入力を優先、なければシークレットから取得） ---
# try:
#     if not GEMINI_API_KEY:
#         GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
# except Exception:
#     pass

# try:
#     if not GITHUB_TOKEN:
#         GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
# except Exception:
#     pass
# # --------------------------------------------------------------------------

# client = None

# # 新しいClient方式での初期化
# if GEMINI_API_KEY:
#     try:
#         client = genai.Client(api_key=GEMINI_API_KEY)
#         print("✅ 最新システム(google.genai)の起動に成功しました！下の各コントロールフォームへ進んでください。")
#     except Exception as e:
#         print(f"【エラー】Geminiの初期化に失敗しました: {e}")
# else:
#     print("【警告】Gemini APIキーが設定されていません。画面から入力するか、Colabのシークレットに 'GEMINI_API_KEY' を登録してください。")


# # ==============================================================================
# # 🧠 内部ロジック関数
# # ==============================================================================
# def _internal_generate(level):
#     if not client:
#         print("❌ Gemini APIキーが設定されていないため、問題を生成できません。")
#         return None

#     history_summary = "なし"
#     if os.path.exists(LOG_FILE):
#         with open(LOG_FILE, "r", encoding="utf-8") as f:
#             try:
#                 logs = json.load(f)
#                 past_topics = []
#                 for day in logs:
#                     for ch in day.get("challenges", []):
#                         past_topics.append(ch.get("topic", ""))
#                 history_summary = ", ".join(past_topics[-5:])
#             except: pass

#     prompt = f"""
#     AIプログラミングメンターとして、学習者の条件（レベル: {level}、最近解いたテーマ: {history_summary}）に合う課題を1問作成してください。
#     【出力フォーマット】必ず以下のJSON形式でのみ出力してください。
#     {{
#         "title": "課題のタイトル",
#         "topic": "学習テーマ",
#         "description": "問題の具体的な説明",
#         "example_input": "入力例",
#         "example_output": "期待される出力例"
#     }}
#     """

#     # 📝 最新の生成APIの書き方に修正（モデル名に「models/」は不要になりました）
#     response = client.models.generate_content(
#         model=MODEL_NAME,
#         contents=prompt,
#         config=types.GenerateContentConfig(
#             response_mime_type="application/json",
#         ),
#     )
#     return json.loads(response.text)

# def _internal_hint(challenge):
#     if not client: return
#     print("🤔 メンターがヒントを考えています...")
#     prompt = f"""
#     学習者が以下の課題で行き詰まっています。解答コードそのものは絶対に明かさず、
#     実装の手がかりとなるヒントを3段階で優しく教えてください。
#     課題: {challenge['title']}\n内容: {challenge['description']}
#     """
#     response = client.models.generate_content(
#         model=MODEL_NAME,
#         contents=prompt,
#     )
#     print(f"\n💡 【メンターからのヒント】\n{response.text}")

# def _internal_review(challenge, user_code):
#     if not client: return
#     print("\n🩺 AIメディックがコードを診断中...")
#     prompt = f"""
#     出題した課題に対して、学習者が書いたコードをレビューし、採点とアドバイスをしてください。
#     【課題】{challenge['title']} / {challenge['description']}
#     【コード】\n```python\n{user_code}\n```
#     【出力フォーマット】必ず以下のJSON形式でのみ出力してください。
#     {{
#         "status": "「クリア」または「要修正」",
#         "score": 100点満点中の点数(数値),
#         "review": "コードのアドバイス",
#         "refactored_code": "60点以上の場合は見本コード。60点未満の場合は空欄"
#     }}
#     """
#     response = client.models.generate_content(
#         model=MODEL_NAME,
#         contents=prompt,
#         config=types.GenerateContentConfig(
#             response_mime_type="application/json",
#         ),
#     )
#     result = json.loads(response.text)

#     print(f"\n📊 採点結果: {result['score']}点 【{result['status']}】")
#     print(f"💬 メンターのアドバイス:\n{result['review']}")

#     if result['score'] < 60:
#         print("\n⚠️ 【判定: 不合格】点数が60点に満たなかったため、今回のコードは記録されません。")
#         return

#     if result['refactored_code']:
#         print(f"\n💡 模範コード（リファクタリング案）:\n{result['refactored_code']}")

#     _internal_save_and_push(challenge, user_code, result['score'])

# def _internal_save_and_push(challenge, user_code, score):
#     today_str = str(datetime.date.today())
#     logs = []
#     if os.path.exists(LOG_FILE):
#         with open(LOG_FILE, "r", encoding="utf-8") as f:
#             try: logs = json.load(f)
#             except: pass

#     challenge_data = {
#         "time": datetime.datetime.now().strftime("%H:%M:%S"),
#         "title": challenge['title'],
#         "topic": challenge['topic'],
#         "problem_description": challenge['description'],
#         "my_code": user_code,
#         "score": score
#     }

#     existing_day_index = -1
#     for i, log in enumerate(logs):
#         if log.get("date") == today_str:
#             existing_day_index = i
#             break

#     if existing_day_index != -1:
#         if not isinstance(logs[existing_day_index], dict) or "challenges" not in logs[existing_day_index]:
#             logs[existing_day_index] = {"date": today_str, "total_solved_today": 0, "challenges": []}
#         logs[existing_day_index]["challenges"].append(challenge_data)
#         logs[existing_day_index]["total_solved_today"] = len(logs[existing_day_index]["challenges"])
#         print(f"\n🔄 本日（{today_str}）の{logs[existing_day_index]['total_solved_today']}問目の合格コードを履歴に追加しました！")
#         is_new_day = False
#     else:
#         new_day_data = {
#             "date": today_str,
#             "total_solved_today": 1,
#             "challenges": [challenge_data]
#         }
#         logs.append(new_day_data)
#         print(f"\n🔥 今日の最初の学習スタンプを記録しました！ (累計クリア日数: {len(logs)}日)")
#         is_new_day = True

#     content_str = json.dumps(logs, ensure_ascii=False, indent=4)
#     with open(LOG_FILE, "w", encoding="utf-8") as f:
#         f.write(content_str)

#     if not GITHUB_TOKEN or GITHUB_USER == "yukisirosui":
#         print("ℹ️ GitHub設定が未完了、または初期設定のままです。履歴はColab内にローカル保存されました。")
#         return

#     try:
#         url = f"https://github.com{GITHUB_USER}/{REPO_NAME}/contents/{LOG_FILE}"
#         headers = {"Authorization": f"token {GITHUB_TOKEN}", "Accept": "application/vnd.github.v3+json"}

#         res = requests.get(url, headers=headers)
#         sha = res.json().get("sha") if res.status_code == 200 else None

#         commit_msg = f"🌱 Add New Day: {today_str}" if is_new_day else f"🚀 Add Another Solved Challenge: {today_str}"
#         data = {
#             "message": f"🤖 {commit_msg} ({challenge['title']})",
#             "content": base64.b64encode(content_str.encode("utf-8")).decode("utf-8")
#         }
#         if sha: data["sha"] = sha
#         put_res = requests.put(url, headers=headers, json=data)

#         if put_res.status_code in [200, 201]:
#             print(f"✨ GitHubとの同期に成功！今日の合格コードがすべて安全に保存されました🌱")
#         else:
#             print(f"⚠️ GitHub同期エラー: {put_res.json().get('message')}")
#     except Exception as e:
#         print(f"❌ GitHub連携中にエラーが発生しました: {e}")


In [16]:
# @title
# #@title 📅 ステップ1: 課題の生成＆ヒントフォーム { display-mode: "form" }
# 現在のレベル = "初級" #@param ["初級", "中級", "上級"]
# ヒントを表示する = False #@param {type:"boolean"}

# import datetime # datetimeモジュールをインポート

# # チャレンジを生成または既存のものを使用
# if not ヒントを表示する: # ヒントを表示しない場合、常に新しい課題を生成
#     today_challenge = _internal_generate(現在のレベル)
# elif 'today_challenge' not in locals(): # ヒントを表示する場合で、まだ課題が生成されていない場合
#     print("⚠️ 課題がまだ生成されていません。新しい課題を生成します。")
#     today_challenge = _internal_generate(現在のレベル)
# # else: ヒントを表示する場合で、すでに課題が生成されている場合は既存のものを使用

# # 常に問題文を表示
# print(f"📅 【本日（{datetime.date.today()}）の課題】")
# print(f"📌 タイトル: {today_challenge['title']} (テーマ: {today_challenge['topic']})")
# print(f"📝 問題内容:\n{today_challenge['description']}")
# if 'example_input' in today_challenge and 'example_output' in today_challenge:
#     print(f"📥 入力例: {today_challenge['example_input']}\n📤 出力例: {today_challenge['example_output']}")
# print("--------------------------------------------------")

# # ヒントを表示するがTrueの場合のみヒントを表示
# if ヒントを表示する:
#     _internal_hint(today_challenge)
#     print("--------------------------------------------------")
# else:
#     print("💡 ヒントが必要な場合は `ヒントを表示する` をTrueにして再度このセルを実行してください。")

In [17]:
# @title
# #@title 💻 ステップ2: 解答提出フォーム { display-mode: "form" }
# #@markdown ---
# #@markdown ### 📝 あなたのPythonコードをここに記述してください
# #@markdown 上記の課題に対する解答コードを、以下のトリプルクォーテーション `"""` の間に直接入力してください。
# #@markdown コードを記述し終えたら、このセルを実行してAIメンターからの診断を受けましょう。
# #@markdown ---
# あなたの回答コード = """
#     x = input("ここに整数を記入してください：")
#     y = input("ここに２つ目の整数を記入してください：")
#     total = int(x) + int(y)
#     print(f"{x} + {y} = {total}")
# """

# if 'today_challenge' in locals():
#     _internal_review(today_challenge, あなたの回答コード)
# else:
#     print("⚠️ 先に『ステップ1の課題生成』を実行してください。")